# Standalone Tuned Random Forest Airfare Predictor (Automated & Multi-Airline)

This notebook implements a streamlined, production-grade **Random Forest Airfare Predictor**:
1. **Simplified Feature Set (No Duration Required):** Flight duration is removed as an input feature—passengers only specify origin, destination, and stop count.
2. **Automated Date & Calendar Intelligence:**
   - The user inputs only the **travelling date** (`YYYY-MM-DD`).
   - The program automatically determines the **current date** and calculates **booking lead time (`days_left`)**.
   - The program automatically extracts **`journey_month`** and evaluates the **weekend travel surcharge (`is_weekend`)**.
3. **Multi-Airline Price Comparison:** The user does not need to enter an airline. The model evaluates and displays estimated fares across **all operating airlines** (e.g., IndiGo, SpiceJet, Akasa Air, Air India, Vistara), ranked from lowest to highest price.
4. **End-to-End Pipeline & Serialization:** Complete pipeline saved to `rf_airfare_model.pkl` for immediate inference.


In [1]:
print("Purpose: Import necessary libraries for data processing, Random Forest modeling, and model serialization.")

import os
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime, date
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("All libraries imported successfully.")


Purpose: Import necessary libraries for data processing, Random Forest modeling, and model serialization.


All libraries imported successfully.


In [2]:
print("Purpose: Ingest dataset, engineer journey month, and extract refined predictor features (excluding duration).")

# Locate dataset whether running from root or app/ directory
csv_path = 'indian_flights.csv' if os.path.exists('indian_flights.csv') else '../indian_flights.csv'
df = pd.read_csv(csv_path)

# Extract journey month to capture seasonal demand cycles
df['journey_month'] = pd.to_datetime(df['flight_date']).dt.month

# Curated predictor features (duration_mins is excluded)
selected_features = [
    'airline',         # Airline brand (used internally to predict each airline's fare)
    'source',          # Origin city
    'destination',     # Destination city
    'cabin_class',     # Economy vs Business
    'departure_time',  # Time of day window
    'total_stops',     # Non-stop, 1-stop, 2-stops
    'days_left',       # Booking lead time in days
    'journey_month',   # Seasonality month
    'is_weekend'       # Weekend travel indicator
]

target = 'price'

print(f"Selected {len(selected_features)} predictor features (duration_mins removed):")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")

# Draw representative sample of 100,000 records for fast, accurate training
sample_df = df[selected_features + [target]].sample(n=100000, random_state=42)
X = sample_df[selected_features]
y = sample_df[target]

print(f"\nDataset ready: {len(X):,} records prepared for training and validation.")


Purpose: Ingest dataset, engineer journey month, and extract refined predictor features (excluding duration).


Selected 9 predictor features (duration_mins removed):
  1. airline
  2. source
  3. destination
  4. cabin_class
  5. departure_time
  6. total_stops
  7. days_left
  8. journey_month
  9. is_weekend

Dataset ready: 100,000 records prepared for training and validation.


In [3]:
print("Purpose: Partition records into 80% training set and 20% held-out test set.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training set:  {len(X_train):,} samples (80%)")
print(f"Held-out test: {len(X_test):,} samples (20%)")


Purpose: Partition records into 80% training set and 20% held-out test set.
Training set:  80,000 samples (80%)
Held-out test: 20,000 samples (20%)


In [4]:
print("Purpose: Construct and train the tuned Random Forest Pipeline without flight duration.")

categorical_cols = ['airline', 'source', 'destination', 'cabin_class', 'departure_time', 'total_stops']
numerical_cols = ['days_left', 'journey_month', 'is_weekend']

# Preprocessor: OneHotEncode categories, Scale numerical features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Tuned Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=80,
    max_depth=18,
    min_samples_split=10,
    random_state=42,
    n_jobs=-1
)

# Unified End-to-End Pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', rf_model)
])

print("Training Tuned Random Forest Pipeline...")
rf_pipeline.fit(X_train, y_train)
print("Training completed successfully!")


Purpose: Construct and train the tuned Random Forest Pipeline without flight duration.
Training Tuned Random Forest Pipeline...


Training completed successfully!


In [5]:
print("Purpose: Evaluate model on held-out test records and print evaluation metrics.")

y_pred = rf_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("=" * 60)
print("       TUNED RANDOM FOREST TEST EVALUATION METRICS")
print("=" * 60)
print(f"  R² Score:                 {r2:.4f} ({r2*100:.2f}% variance explained)")
print(f"  Mean Absolute Error (MAE): ₹{mae:,.2f}")
print(f"  Root Mean Squared Error:   ₹{rmse:,.2f}")
print(f"  Mean Absolute % Error:     {mape:.2f}%")
print("=" * 60)


Purpose: Evaluate model on held-out test records and print evaluation metrics.
       TUNED RANDOM FOREST TEST EVALUATION METRICS
  R² Score:                 0.9586 (95.86% variance explained)
  Mean Absolute Error (MAE): ₹431.51
  Root Mean Squared Error:   ₹953.18
  Mean Absolute % Error:     6.82%


In [6]:
print("Purpose: Save the complete trained Random Forest Pipeline to a .pkl file.")

model_filename = 'rf_airfare_model.pkl'
joblib.dump(rf_pipeline, model_filename)

# Also ensure copy in app/ directory if running from root, or root if in app/
if os.path.exists('app'):
    joblib.dump(rf_pipeline, os.path.join('app', model_filename))
elif os.path.exists('../rf_airfare_model.pkl') or os.path.exists('..'):
    try:
        joblib.dump(rf_pipeline, os.path.join('..', model_filename))
    except Exception:
        pass

file_size_mb = os.path.getsize(model_filename) / (1024 * 1024)
print(f"Model pipeline successfully saved to '{model_filename}'.")
print(f"Saved file size: {file_size_mb:.2f} MB")


Purpose: Save the complete trained Random Forest Pipeline to a .pkl file.
Model pipeline successfully saved to 'rf_airfare_model.pkl'.
Saved file size: 64.98 MB


In [7]:
print("Purpose: Reload the saved .pkl model from disk to verify standalone persistence.")

loaded_rf_model = joblib.load('rf_airfare_model.pkl')
print("Model loaded successfully from 'rf_airfare_model.pkl'. Ready for inference!")


Purpose: Reload the saved .pkl model from disk to verify standalone persistence.


Model loaded successfully from 'rf_airfare_model.pkl'. Ready for inference!


In [ ]:
print("Purpose: Accept user travel details, automatically compute booking lead time and weekend flags, and suggest prices across all available airlines.")

# =============================================================================
# --- USER DEFINED FLIGHT DETAILS ---
# Edit the parameters below to test any flight (no duration or airline needed!):
# =============================================================================
user_input = {
    'travel_date': '2026-11-28',             # Travelling Date in YYYY-MM-DD format
    'source': 'Delhi',                       # 'Delhi', 'Mumbai', 'Bengaluru', 'Hyderabad', 'Kolkata', 'Chennai', 'Cochin'
    'destination': 'Mumbai',                 # 'Delhi', 'Mumbai', 'Bengaluru', 'Hyderabad', 'Ahmedabad', 'Goa', 'Cochin'
    'cabin_class': 'Economy',                # 'Economy' or 'Business'
    'departure_time': 'Morning (6AM-12PM)',  # 'Early Morning (12AM-6AM)', 'Morning (6AM-12PM)', 'Afternoon (12PM-6PM)', 'Evening (6PM-12AM)'
    'total_stops': 'non-stop'                # 'non-stop', '1-stop', '2-stops'
}

# 1. Automatic Determination of Current Date and Days Left
current_date = date.today()
travel_dt = datetime.strptime(user_input['travel_date'], "%Y-%m-%d").date()
raw_days_left = (travel_dt - current_date).days

# Handle same-day/past dates or clamp within model valid boundary
if raw_days_left <= 0:
    days_left = 1
    lead_note = f"Same-day/Last-minute (Raw difference: {raw_days_left} days)"
else:
    days_left = min(raw_days_left, 90)
    lead_note = f"{raw_days_left} days in advance"

# 2. Automatic Determination of Month and Weekend
journey_month = travel_dt.month
# In this dataset, Friday (4) and Sunday (6) capture peak weekend travel surcharges
is_weekend = 1 if travel_dt.weekday() in [4, 6] else 0
day_name = travel_dt.strftime('%A')

# 3. Determine Available Airlines for Selected Cabin Class
if user_input['cabin_class'] == 'Business':
    # Only Air India and Vistara offer Business Class cabins
    available_airlines = ['Air India', 'Vistara']
else:
    available_airlines = ['Air India', 'Akasa Air', 'IndiGo', 'SpiceJet', 'Vistara']

# 4. Generate Predictions Across All Airlines
fare_estimates = []
for airline in available_airlines:
    flight_record = pd.DataFrame([{
        'airline': airline,
        'source': user_input['source'],
        'destination': user_input['destination'],
        'cabin_class': user_input['cabin_class'],
        'departure_time': user_input['departure_time'],
        'total_stops': user_input['total_stops'],
        'days_left': days_left,
        'journey_month': journey_month,
        'is_weekend': is_weekend
    }])
    
    pred_fare = float(loaded_rf_model.predict(flight_record)[0])
    tier = "Full-Service Carrier" if airline in ['Air India', 'Vistara'] else "Low-Cost Carrier"
    
    fare_estimates.append({
        'Airline': airline,
        'Estimated Fare (₹)': round(pred_fare, 2),
        'Carrier Type': tier
    })

# Format into a clean comparison table sorted by price
fare_df = pd.DataFrame(fare_estimates).sort_values(by='Estimated Fare (₹)').reset_index(drop=True)
fare_df['Rank'] = [f"#{i+1}" for i in range(len(fare_df))]
fare_df = fare_df[['Rank', 'Airline', 'Estimated Fare (₹)', 'Carrier Type']]

# Display Clean Flight Summary and Airline Price Comparison
print("=" * 80)
print("                   AIRFARE ESTIMATION ACROSS ALL AIRLINES")
print("=" * 80)
print(f"  Origin -> Destination:  {user_input['source']} -> {user_input['destination']}")
print(f"  Travel Date:            {user_input['travel_date']} ({day_name})")
print(f"  Current Date:           {current_date}")
print(f"  Booking Lead Time:      {lead_note}")
print(f"  Journey Month:          Month {journey_month} ({datetime(2000, journey_month, 1).strftime('%B')})")
print(f"  Weekend Surge:          {'Yes (Friday/Sunday surge applies)' if is_weekend else 'No (Standard weekday pricing)'}")
print(f"  Cabin Class:            {user_input['cabin_class']}")
print(f"  Flight Configuration:   {user_input['total_stops']} | {user_input['departure_time']}")
print("-" * 80)
print("  PREDICTED FARES BY AIRLINE (RANKED BY BEST VALUE):")
print("-" * 80)
display(fare_df)
print("=" * 80)


Purpose: Accept user travel details, automatically compute booking lead time and weekend flags, and suggest prices across all available airlines.
                   AIRFARE ESTIMATION ACROSS ALL AIRLINES
  Origin -> Destination:  Delhi -> Mumbai
  Travel Date:            2026-11-28 (Saturday)
  Current Date:           2026-11-20
  Booking Lead Time:      8 days in advance
  Journey Month:          Month 11 (November)
  Weekend Surge:          No (Standard weekday pricing)
  Cabin Class:            Economy
  Flight Configuration:   non-stop | Morning (6AM-12PM)
--------------------------------------------------------------------------------
  PREDICTED FARES BY AIRLINE (RANKED BY BEST VALUE):
--------------------------------------------------------------------------------


,Rank,Airline,Estimated Fare (₹),Carrier Type
0,#1,Akasa Air,6859.29,Low-Cost Carrier
1,#2,SpiceJet,6864.26,Low-Cost Carrier
2,#3,IndiGo,6912.70,Low-Cost Carrier
3,#4,Air India,8324.57,Full-Service Carrier
4,#5,Vistara,9069.16,Full-Service Carrier
